# GraphRAG: AI Copyright and Governance Knowledge Graph

This notebook builds a full *GraphRAG pipleline* on a pre-scraped dataset of news articles about AI Copyright, governance, and intellectual property.

## 0. Install dependencies

In [1]:
# graspologic has no stable release for Python 3.13 (all <= 3.4.4 cap at
# requires_python <3.13, so pip backtracks to 0.3.1 and fails building gensim).
# 3.4.5.dev2 lifts the cap to <3.15. The exact pre-release pin is honoured
# without --pre, so everything else still resolves to stable versions.
!pip install llama-index llama-index-llms-google-genai "graspologic==3.4.5.dev2" pandas nest-asyncio python-dotenv networkx

# 1. Imports

In [2]:
import os
import asyncio
import nest_asyncio2
import pandas as pd
import networkx as nx
from dotenv import load_dotenv

from llama_index.core import Document, PropertyGraphIndex, Settings
from llama_index.core.graph_stores.types import (
    EntityNode, KG_NODES_KEY, KG_RELATIONS_KEY, Relation
)
from llama_index.core.graph_stores import SimplePropertyGraphStore
from llama_index.core.llms.llm import LLM
from llama_index.core.prompts import PromptTemplate
from llama_index.core.schema import TransformComponent, BaseNode
from llama_index.core.async_utils import run_jobs
from llama_index.core.query_engine import CustomQueryEngine
from llama_index.llms.google_genai import GoogleGenAI
from graspologic.partition import hierarchical_leiden

# Patch the event loop so LlamaIndex async calls work inside Jupyter
nest_asyncio2.apply()

# Load API keys from .env file
load_dotenv()

# Gemini config
api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    raise ValueError("Set GEMINI_API_KEY in your .env file")

# Set the default LLM for LlamaIndex components
Settings.llm = GoogleGenAI(
    model="models/gemini-3.5-flash-lite",
    api_key=api_key,
)

print("✅ Gemini imports ready")

✅ Gemini imports ready


# 2. Configuration

In [3]:
# ── LLM Configuration ─────────────────────────────────────────────────────────
# Gemini equivalents of the source's OpenAI pair: flash-lite is the cheap
# high-volume tier (gpt-4o-mini's role), flash the stronger one (gpt-4o's role).
EXTRACTION_LLM = GoogleGenAI(model="models/gemini-3.5-flash-lite", api_key=api_key, temperature=0)  # For extraction + summaries
QUERY_LLM      = GoogleGenAI(model="models/gemini-3.5-flash",      api_key=api_key, temperature=0)  # For final answer synthesis

# ── Pipeline Parameters ────────────────────────────────────────────────────────
MAX_ARTICLES        = 50    # Number of articles to process
MAX_PATHS_PER_CHUNK = 20    # Max entity-relationship-entity triplet|s extracted per chunk
NUM_WORKERS         = 4     # Parallel workers for extraction (faster, same cost)
MAX_CLUSTER_SIZE    = 10    # Max entities per community cluster (controls how big the clusters should be)
GRAPH_OUTPUT_FILE   = "ai_copyright_graph.html"

Settings.llm = EXTRACTION_LLM

print(f"✅ Using {EXTRACTION_LLM.model} for extraction, {QUERY_LLM.model} for querying")

✅ Using models/gemini-3.5-flash-lite for extraction, models/gemini-3.5-flash for querying


# 3. Define the Ontology

In [4]:
# ── Entity types ──────────────────────────────────────────────────────────────
# We'll embed these directly into the extraction prompt below.

ENTITY_TYPES = [
    "ORGANIZATION",  # Companies, labs, industry groups (OpenAI, Google, RIAA)
    "PERSON",        # Executives, policymakers, judges (Sam Altman, Thierry Breton)
    "LEGISLATION",   # Laws, acts, regulations (EU AI Act, DMCA, Copyright Act)
    "LEGAL_CASE",    # Lawsuits, court rulings (NYT v. OpenAI, Getty v. Stability AI)
    "CONCEPT",       # Abstract ideas: fair use, training data, IP rights
    "GOVERNMENT",    # Nations, regulatory bodies, courts (EU, US Copyright Office)
    "AI_SYSTEM",     # Specific models or products (GPT-4, Stable Diffusion, Gemini)
]

# ── Relationship types ─────────────────────────────────────────────────────────
RELATION_TYPES = [
    "FILED_AGAINST",  # Plaintiff Organization/Person → LEGAL_CASE
    "DEFENDANT_IN",   # Organization/Person → LEGAL_CASE
    "REGULATES",      # GOVERNMENT/LEGISLATION → ORGANIZATION/AI_SYSTEM
    "ADVOCATES_FOR",  # ORGANIZATION/PERSON → CONCEPT or policy position
    "TRAINED_ON",     # AI_SYSTEM → CONCEPT or dataset type
    "PART_OF",        # PERSON → ORGANIZATION
    "REFERENCES",     # LEGAL_CASE/LEGISLATION → CONCEPT
    "OPPOSES",        # ORGANIZATION/GOVERNMENT → LEGISLATION/CONCEPT
]

print("✅ Ontology:")
print(f"   Entity types:       {ENTITY_TYPES}")
print(f"   Relationship types: {RELATION_TYPES}")

✅ Ontology:
   Entity types:       ['ORGANIZATION', 'PERSON', 'LEGISLATION', 'LEGAL_CASE', 'CONCEPT', 'GOVERNMENT', 'AI_SYSTEM']
   Relationship types: ['FILED_AGAINST', 'DEFENDANT_IN', 'REGULATES', 'ADVOCATES_FOR', 'TRAINED_ON', 'PART_OF', 'REFERENCES', 'OPPOSES']


# 4. Extraction Prompt

In [5]:
# Build the entity and relationship type strings dynamically from our ontology
entity_types_str   = ", ".join(ENTITY_TYPES)
relation_types_str = ", ".join(RELATION_TYPES)

KG_TRIPLET_EXTRACT_TMPL = f"""
-Goal-
Given a news article about AI copyright, governance, or intellectual property,
identify all entities mentioned in the article and their relationships.

Extract up to {{max_knowledge_triplets}} entity-relation triplets.

-Allowed Entity Types-
{entity_types_str}

-Allowed Relationship Types-
{relation_types_str}

-Steps-
1. Identify ALL entities. For each entity extract:
   - name: Name of the entity, capitalized
   - type: One of the allowed entity types above
   - description: A brief description of the entity and its role in AI copyright/governance

2. Identify relationships between entities. For each pair extract:
   - source: name of the source entity
   - target: name of the target entity
   - relation: one of the allowed relationship types above
   - description: a sentence explaining why and how these entities are related

-Real Data-
######################
text: {{text}}
######################
"""

print("✅ Extraction prompt ready")
print(f"\nPreview (first 300 chars):\n{KG_TRIPLET_EXTRACT_TMPL[:300]}...")

✅ Extraction prompt ready

Preview (first 300 chars):

-Goal-
Given a news article about AI copyright, governance, or intellectual property,
identify all entities mentioned in the article and their relationships.

Extract up to {max_knowledge_triplets} entity-relation triplets.

-Allowed Entity Types-
ORGANIZATION, PERSON, LEGISLATION, LEGAL_CASE, CONC...


# 5. Pydantic Extraction Models

In [6]:
from pydantic import BaseModel, Field, field_validator
from typing import Literal, List

EntityTypeStr = Literal[
    "ORGANIZATION", "PERSON", "LEGISLATION", "LEGAL_CASE",
    "CONCEPT", "GOVERNMENT", "AI_SYSTEM"
]
RelationTypeStr = Literal[
    "FILED_AGAINST", "DEFENDANT_IN", "REGULATES", "ADVOCATES_FOR",
    "TRAINED_ON", "PART_OF", "REFERENCES", "OPPOSES"
]

class ExtractedEntity(BaseModel):
    name: str = Field(description="Name of the entity, capitalized")
    type: EntityTypeStr = Field(description="One of the allowed entity types")
    description: str = Field(description="Brief description of the entity and its role")

class ExtractedRelationship(BaseModel):
    source: str = Field(description="Name of the source entity")
    target: str = Field(description="Name of the target entity")
    relation: RelationTypeStr = Field(description="One of the allowed relationship types")
    description: str = Field(description="Sentence explaining the relationship")

class ExtractionResult(BaseModel):
    entities: List[ExtractedEntity] = Field(default_factory=list)
    relationships: List[ExtractedRelationship] = Field(default_factory=list)

print("✅ Pydantic extraction models defined")

✅ Pydantic extraction models defined


# 6. GraphRAG Extractor

In [7]:
class GraphRAGExtractor(TransformComponent):
    """
    Extracts entities and relationships WITH descriptions from each text chunk.

    Uses Pydantic structured output (via OpenAI function calling) to guarantee
    that every entity has a valid type and every relationship uses an allowed label.

    Data flow for a single chunk:
      Raw text
        → LLM (astructured_predict)
        → ExtractionResult(
              entities=[
                  ExtractedEntity(name="OpenAI", type="ORGANIZATION", description="AI lab..."),
                  ExtractedEntity(name="NYT v. OpenAI", type="LEGAL_CASE", description="Copyright lawsuit..."),
              ],
              relationships=[
                  ExtractedRelationship(source="OpenAI", target="NYT v. OpenAI",
                                        relation="DEFENDANT_IN", description="OpenAI is named defendant..."),
              ]
          )
        → EntityNode(name="OpenAI", label="ORGANIZATION", properties={...})
        → EntityNode(name="NYT v. OpenAI", label="LEGAL_CASE", properties={...})
        → Relation(label="DEFENDANT_IN", source_id=<OpenAI id>, target_id=<NYT v. OpenAI id>)
    """

    llm: LLM = Field(default_factory=lambda: Settings.llm)
    extract_prompt: PromptTemplate = Field(default_factory=lambda: PromptTemplate(KG_TRIPLET_EXTRACT_TMPL))
    num_workers: int = 4
    max_paths_per_chunk: int = 10

    @field_validator("extract_prompt", mode="before")
    @classmethod
    def coerce_to_prompt_template(cls, v):
        return PromptTemplate(v) if isinstance(v, str) else v

    def __call__(self, nodes, show_progress=False, **kwargs):
        return asyncio.run(self.acall(nodes, show_progress=show_progress, **kwargs))

    async def _aextract(self, node: BaseNode) -> BaseNode:
        text = node.get_content(metadata_mode="llm")

        # ── Step 1: Call the LLM ───────────────────────────────────────────────
        # astructured_predict uses OpenAI function calling to return a validated ExtractionResult format
        # so entity types and relation labels are guaranteed valid.
        try:
            result = await self.llm.astructured_predict(
                ExtractionResult,
                self.extract_prompt,
                text=text,
                max_knowledge_triplets=self.max_paths_per_chunk,
            )
            entities      = result.entities
            relationships = result.relationships
            
        except Exception as e:
            print(f"Extraction error: {e}")
            entities, relationships = [], []

        existing_nodes     = node.metadata.pop(KG_NODES_KEY, [])
        existing_relations = node.metadata.pop(KG_RELATIONS_KEY, [])
        base_metadata      = node.metadata.copy()

        # ── Step 2: Entities → EntityNode objects ─────────────────────────────
        # e.g. ExtractedEntity(name="OpenAI", type="ORGANIZATION", description="...")
        # becomes  EntityNode(name="OpenAI", label="ORGANIZATION", properties={...})
        existing_nodes += [
            EntityNode(
                name=entity.name,
                label=entity.type,
                properties={**base_metadata, "entity_description": entity.description},
            )
            for entity in entities
        ]

        # ── Step 3: Relationships → Relation objects ───────────────────────────
        # entity_lookup lets us attach the correct label to relationship source and target nodes.
        # e.g. {"OpenAI": "ORGANIZATION", "NYT v. OpenAI": "LEGAL_CASE"}
        entity_lookup = {e.name: e.type for e in entities}

        for rel in relationships:
            source_node = EntityNode(
                name=rel.source,
                label=entity_lookup.get(rel.source, "ENTITY"),
                properties=base_metadata,
            )
            target_node = EntityNode(
                name=rel.target,
                label=entity_lookup.get(rel.target, "ENTITY"),
                properties=base_metadata,
            )
            # Only add endpoint nodes not already captured from the entity pass
            if rel.source not in entity_lookup:
                existing_nodes.append(source_node)
            if rel.target not in entity_lookup:
                existing_nodes.append(target_node)
            existing_relations.append(Relation(
                label=rel.relation,
                source_id=source_node.id,
                target_id=target_node.id,
                properties={**base_metadata, "relationship_description": rel.description},
            ))

        node.metadata[KG_NODES_KEY]     = existing_nodes
        node.metadata[KG_RELATIONS_KEY] = existing_relations
        return node

    async def acall(self, nodes, show_progress=False, **kwargs):
        """Process all nodes in parallel using num_workers async workers."""
        jobs = [self._aextract(node) for node in nodes]
        return await run_jobs(
            jobs,
            workers=self.num_workers,
            show_progress=show_progress,
            desc="Extracting triplets",
        )

print("✅ GraphRAGExtractor defined")

✅ GraphRAGExtractor defined


# 7. GraphRAG Store

In [8]:
class GraphRAGStore(SimplePropertyGraphStore):
    """
    Extends SimplePropertyGraphStore with:
    - Leiden community detection
    - LLM-generated community summaries (using entity + relationship descriptions)

    After building the PropertyGraphIndex, call build_communities() to
    detect clusters and generate summaries. These summaries are then
    used by GraphRAGQueryEngine at query time.
    """

    community_summaries: dict = {}

    def build_communities(self):
        """
        Main entry point: run community detection and generate summaries.
        Call this once after PropertyGraphIndex is built.
        """
        print("Running community detection...")
        nx_graph = self._to_networkx()

        if not nx_graph.nodes:
            print("⚠️  Graph is empty — no communities to detect")
            return

        print(f"Graph has {nx_graph.number_of_nodes()} nodes, {nx_graph.number_of_edges()} edges")

        # Run hierarchical Leiden algorithm
        clusters = hierarchical_leiden(nx_graph, max_cluster_size=MAX_CLUSTER_SIZE)
        num_communities = len(set(c.cluster for c in clusters))
        print(f"Found {num_communities} communities")

        # Collect entities and relationships per community
        community_info = self._collect_community_info(nx_graph, clusters)

        # Generate LLM summaries
        self._generate_summaries(community_info)
        print(f"\n✅ {len(self.community_summaries)} community summaries generated")

    def _to_networkx(self) -> nx.Graph:
        """
        Convert the LlamaIndex property graph to a NetworkX graph for Graspologic.
        Edges carry relationship label and description as attributes.
        """
        nx_graph = nx.Graph()

        # Add entity nodes
        for node in self.graph.nodes.values():
            if isinstance(node, EntityNode):
                nx_graph.add_node(node.id)

        # Add edges with relationship metadata
        for relation in self.graph.relations.values():
            if relation.source_id in nx_graph and relation.target_id in nx_graph:
                nx_graph.add_edge(
                    relation.source_id,
                    relation.target_id,
                    relationship=relation.label,
                    description=relation.properties.get("relationship_description", ""),
                )

        return nx_graph

    def _collect_community_info(self, nx_graph, clusters) -> dict:
        """
        For each community cluster, collect:
        - Entity names, types, and descriptions
        - Relationship strings (with descriptions where available)

        This rich data is what makes the LLM summaries informative.
        """
        community_mapping = {item.node: item.cluster for item in clusters}

        # Build a lookup of node id -> {name, type, description}
        node_details = {}
        for node in self.graph.nodes.values():
            if not isinstance(node, EntityNode):
                continue
            node_details[node.id] = {
                "name":        node.name,
                "type":        node.label,
                "description": node.properties.get("entity_description", ""),
            }

        community_info = {}
        for item in clusters:
            cid, nid = item.cluster, item.node
            community_info.setdefault(cid, {"entities": [], "relationships": []})

            # Add entity details to community
            if nid in node_details:
                community_info[cid]["entities"].append(node_details[nid])

            # Add relationships to neighbors in the same community
            for neighbor in nx_graph.neighbors(nid):
                if community_mapping.get(neighbor) == cid:
                    edge = nx_graph.get_edge_data(nid, neighbor)
                    rel  = edge.get("relationship", "RELATED") if edge else "RELATED"
                    desc = edge.get("description",  "")        if edge else ""

                    src_name = node_details.get(nid,      {}).get("name", nid)
                    tgt_name = node_details.get(neighbor, {}).get("name", neighbor)

                    # Format: "Entity A --[RELATION]--> Entity B (description)"
                    entry = f"{src_name} --[{rel}]--> {tgt_name}"
                    if desc:
                        entry += f" ({desc})"
                    community_info[cid]["relationships"].append(entry)

        return community_info

    def _generate_summaries(self, community_info):
        """
        Ask the LLM to write a briefing note for each community cluster.
        Uses entity descriptions and relationship descriptions for richer context.
        """
        for community_id, data in community_info.items():
            if not data["relationships"] and not data["entities"]:
                continue

            # Format entities with type and description
            entities_text = "\n".join([
                f"- {e['name']} ({e['type']}): {e['description']}"
                for e in data["entities"] if e.get("name")
            ])

            # Deduplicate and format relationships
            relationships_text = "\n".join(sorted(set(data["relationships"])))

            prompt = f"""You are analysing a cluster of entities from news articles about
            AI copyright, governance, and intellectual property.

            Entities in this cluster:
            {entities_text}

            Relationships:
            {relationships_text}

            Write a concise briefing (3-5 sentences) that:
            1. Identifies the main organizations, people, legal cases, or topics in this cluster
            2. Explains how they are connected and why — including legal or regulatory context
            3. Highlights any disputes, lawsuits, policy positions, or tensions
            4. Notes anything particularly relevant for understanding AI copyright or governance

            Briefing:"""

            response = EXTRACTION_LLM.complete(prompt)
            self.community_summaries[community_id] = response.text
            print(f"  Community {community_id}: {response.text[:100]}...")

    def get_community_summaries(self) -> dict:
        return self.community_summaries

print("✅ GraphRAGStore defined")

✅ GraphRAGStore defined


# 8. GraphRAG Query Engine

In [9]:
class GraphRAGQueryEngine(CustomQueryEngine):
    """
    Queries all community summaries and synthesises a single answer.

    Step 1: For each community summary, ask EXTRACTION_LLM to answer the
            question based only on that summary. If not relevant, skip.
    Step 2: Aggregate all relevant partial answers using QUERY_LLM into
            one final, clear, non-redundant response.
    """

    graph_store: GraphRAGStore
    llm: LLM

    def custom_query(self, query_str: str) -> str:
        summaries = self.graph_store.get_community_summaries()

        if not summaries:
            return "No community summaries found. Run graph_store.build_communities() first."

        # Step 1: Get a partial answer from each community summary
        community_answers = [
            self._answer_from_community(summary, query_str)
            for summary in summaries.values()
        ]

        # Filter out empty/irrelevant responses
        relevant_answers = [a for a in community_answers if a.strip()]

        if not relevant_answers:
            return "I don't have enough information in the knowledge graph to answer that question."

        # Step 2: Aggregate into one final answer
        return self._aggregate(relevant_answers, query_str)

    def _answer_from_community(self, summary: str, query: str) -> str:
        """
        Ask EXTRACTION_LLM to answer the query from a single community summary.
        Returns empty string if the summary isn't relevant to the question.
        We use the cheaper model here since this runs once per community.
        """
        prompt = (
            f"Community summary:\n{summary}\n\n"
            f"Question: {query}\n\n"
            f"If this summary contains information relevant to the question, answer it. "
            f"If not relevant, reply exactly: 'No relevant information.'\n\n"
            f"Answer:"
        )
        response = EXTRACTION_LLM.complete(prompt)
        text = response.text.strip()

        # Filter out non-answers
        return "" if "no relevant information" in text.lower() else text

    def _aggregate(self, answers: List[str], query: str) -> str:
        """
        Synthesise all relevant partial answers into one final response.
        Uses QUERY_LLM (gpt-4o) for better reasoning quality on the final step.
        """
        combined = "\n\n---\n\n".join(answers)
        prompt = (
            f"You have received answers from multiple knowledge graph communities about this question:\n\n"
            f"Question: {query}\n\n"
            f"Community answers:\n{combined}\n\n"
            f"Synthesise these into a single, clear, well-structured final answer. "
            f"Remove redundancy, keep all important details, and ensure the answer "
            f"directly addresses the question.\n\n"
            f"Final Answer:"
        )
        return self.llm.complete(prompt).text

print("✅ GraphRAGQueryEngine defined")

✅ GraphRAGQueryEngine defined


# 9. Load Article Dataset

In [10]:
# Load from CSV ───────────────────────────────────────────────────
df = pd.read_csv("ai_copyright_dataset.csv")
print(f"Number of articles: {len(df)}")
df.head()

Number of articles: 10


,query,title,snippet,source,date,url,type,full_text,video_id,status
0,AI Intellectual Property,"AI, Copyright, and the Law: The Ongoing Battle...",Artificial intelligence (AI) is rapidly reshap...,University of Southern California,"Feb 4, 2025",https://sites.usc.edu/iptls/2025/02/04/ai-copy...,article,By: Negar Bondari\nArtificial intelligence (AI...,NaN,success
1,AI Intellectual Property,Artificial Intelligence and Intellectual Property,AI inventions present the current patent syste...,World Intellectual Property Organization (WIPO),NaN,https://www.wipo.int/en/web/frontier-technolog...,article,Artificial Intelligence and Intellectual Prope...,NaN,success
2,AI Intellectual Property,Generative AI: Navigating intellectual property,Artificial intelligence challenges the traditi...,Nixon Peabody,"Sep 17, 2025",https://www.nixonpeabody.com/insights/articles...,article,Generative AI is transforming creative and tec...,NaN,success
3,AI Intellectual Property,Intellectual Property Rights and AI-Generated ...,U.S. law does allow humans to claim copyright ...,"Medium · Adnan Masood, PhD.",NaN,https://medium.com/@adnanmasood/intellectual-p...,article,Member-only story\nIntellectual Property Right...,NaN,success
4,Copyright Generative AI,Copyright and Artificial Intelligence | U.S. C...,Copyright and Artificial Intelligence analyzes...,Copyright Office (.gov),NaN,https://www.copyright.gov/ai/,article,Copyright and Artificial Intelligence\nSince l...,NaN,success


# 10. Wrap articles as documents

In [11]:
# Wrap each article as a LlamaIndex Document
# full_text is the main content; everything else is metadata
nodes = [
    Document(
        text=str(row["full_text"]),
        metadata={
            "title":  str(row.get("title",  "")),
            "source": str(row.get("source", "")),
            "date":   str(row.get("date",   "")),
        }
    )
    for _, row in df.iterrows()
]

print(f"✅ Created {len(nodes)} LlamaIndex documents, stored as nodes")

✅ Created 10 LlamaIndex documents, stored as nodes


In [12]:
print(nodes[3].text)

Member-only story
Intellectual Property Rights and AI-Generated Content — Issues in Human Authorship, Fair Use Doctrine, and Output Liability
Critical IP Questions Every Leader Needs to Answer for Future Readiness — Analysis of Copyright Law Implications for Generative Artificial Intelligence — Comparative Legal Frameworks, Authorship Paradigms, and Training Data Considerations
Generative Artificial Intelligence is rapidly reshaping industries, but its collision with long-established copyright law presents critical strategic questions that leaders ignore at their peril. For leaders charting a course through the AI revolution, navigating the intricate maze of copyright law is no longer a niche legal concern but a core strategic imperative impacting innovation, risk, and global competitiveness.
In my discussions with leadership across various sectors, the rapid rise of generative AI consistently brings up critical questions regarding intellectual property and copyright. These aren’t just

# 11. Build the Knnowledge Graph

In [13]:
# Instantiate the extractor with our ontology prompt
kg_extractor = GraphRAGExtractor(
    llm=EXTRACTION_LLM,
    extract_prompt=KG_TRIPLET_EXTRACT_TMPL,
    max_paths_per_chunk=MAX_PATHS_PER_CHUNK,
    num_workers=NUM_WORKERS,
)

# GraphRAGStore is both the graph database and the community detection engine
graph_store = GraphRAGStore()

print("Building knowledge graph... (this may take a few minutes)")

index = PropertyGraphIndex(
    nodes=nodes,
    kg_extractors=[kg_extractor],
    property_graph_store=graph_store,
    embed_kg_nodes=False,             # we don't need vector similarity search for this GraphRAG pipeline
    show_progress=True,
)

Building knowledge graph... (this may take a few minutes)


Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Direct use of automatic function calling (AFC) in AsyncModels.generate_content is not recommended. Instead, we recommend to use AFC in AsyncChat.send_message. Similarly, direct use of AFC in AsyncModels.generate_content_stream is not recommended. Instead, we recommend to use AFC in AsyncChat.send_message_stream.


# 12. Entity Resolution

The extractor sees each chunk independently, so the same real-world entity turns up
under several surface forms — `ChatGPT`/`Chatgpt`, `OpenAI`/`Openai`, `WIPO`/`World
Intellectual Property Organization`. Left alone these become separate graph nodes,
splitting an entity's edges across duplicates and weakening the Leiden communities.

This step folds them together **before** community detection runs.

In [21]:
import re
import unicodedata
from collections import Counter, defaultdict

# Acronyms and alternate surface forms that normalisation alone cannot connect.
# Keys are *normalised* variants; the value is the display name to collapse into.
# Extend this as you notice new duplicates in the entity list below.
ALIAS_MAP = {
    "wipo":                                     "World Intellectual Property Organization",
    "world intellectual property organization": "World Intellectual Property Organization",
    "copyright office":                         "U.S. Copyright Office",
    "us copyright office":                      "U.S. Copyright Office",
    "generative artificial intelligence":       "Generative AI",
    "generative ai":                            "Generative AI",
    "fair use doctrine":                        "Fair Use",
    "fair use":                                 "Fair Use",
    "eu ai act":                                "AI Act",
    "ai act":                                   "AI Act",
}

_LEADING_ARTICLE = re.compile(r"^(?:the|a|an)\s+")


def normalise_entity_name(name: str) -> str:
    """Fold a raw name to a comparison key: 'ChatGPT'/'Chatgpt' -> 'chatgpt'."""
    s = unicodedata.normalize("NFKD", name)
    s = s.encode("ascii", "ignore").decode()   # drop accents
    s = s.lower()
    s = re.sub(r"[.'’]", "", s)           # 'U.S.' -> 'us', 'Thaler v.' -> 'thaler v'
    s = re.sub(r"[^a-z0-9]+", " ", s).strip()
    return _LEADING_ARTICLE.sub("", s)


def resolution_key(name: str) -> str:
    """Group key: the alias target's key if aliased, else the normalised name."""
    norm  = normalise_entity_name(name)
    alias = ALIAS_MAP.get(norm)
    return normalise_entity_name(alias) if alias else norm


def _tidy(name: str) -> str:
    # Legal citations conventionally lowercase the 'v.' ("Thaler v. Perlmutter"),
    # which the uppercase preference below would otherwise clobber.
    return re.sub(r"(?<=\s)V\.(?=\s)", "v.", name)


def _pick_display_name(counts: Counter) -> str:
    """Deterministic pick: most frequent, then most uppercase (keeps acronyms like
    'ChatGPT' over 'Chatgpt'), then longest, then lexicographic as a tie-break."""
    return max(counts, key=lambda n: (counts[n], sum(c.isupper() for c in n), len(n), n))


def resolve_entities(graph_store, verbose=True):
    """
    Merge duplicate entities in place.

    EntityNode.id is derived from its name, so folding names to a canonical form
    is what actually merges nodes. The store keys nodes, relations AND a triplets
    set off those ids, so all three are rewritten together.
    """
    graph = graph_store.graph

    entity_nodes = [n for n in graph.nodes.values() if isinstance(n, EntityNode)]
    other_nodes  = {nid: n for nid, n in graph.nodes.items() if not isinstance(n, EntityNode)}

    groups = defaultdict(list)
    for node in entity_nodes:
        groups[resolution_key(node.name)].append(node)

    id_map, merged_nodes, merges = {}, {}, []
    for members in groups.values():
        counts = Counter(m.name for m in members)

        # An explicit alias target wins; otherwise score the observed surface forms.
        alias_target = next(
            (ALIAS_MAP[normalise_entity_name(m.name)] for m in members
             if normalise_entity_name(m.name) in ALIAS_MAP),
            None,
        )
        display = _tidy(alias_target or _pick_display_name(counts))

        # Prefer a real ontology type over the generic "ENTITY" fallback.
        labels = Counter(m.label for m in members if m.label.upper() != "ENTITY")
        label  = labels.most_common(1)[0][0] if labels else "ENTITY"

        # Keep every distinct description — they feed the community summaries.
        descriptions = []
        for m in members:
            d = (m.properties or {}).get("entity_description", "")
            if d and d not in descriptions:
                descriptions.append(d)

        properties = dict(members[0].properties or {})
        properties["entity_description"] = " | ".join(descriptions)
        if len(counts) > 1:
            properties["aliases"] = sorted(counts)
            merges.append((display, sorted(counts)))

        merged = EntityNode(name=display, label=label, properties=properties)
        merged_nodes[merged.id] = merged
        for m in members:
            id_map[m.id] = merged.id

    # Rewrite relations onto canonical ids, dropping self-loops created by merging
    new_relations, new_triplets, self_loops = {}, set(), 0
    for rel in graph.relations.values():
        source = id_map.get(rel.source_id, rel.source_id)
        target = id_map.get(rel.target_id, rel.target_id)
        if source == target:                       # e.g. OpenAI --[REFERENCES]--> Openai
            self_loops += 1
            continue
        merged_rel = Relation(
            label=rel.label, source_id=source, target_id=target, properties=rel.properties
        )
        new_relations[graph._get_relation_key(relation=merged_rel)] = merged_rel
        new_triplets.add((source, rel.label, target))

    before_nodes, before_rels = len(entity_nodes), len(graph.relations)
    graph.nodes     = {**other_nodes, **merged_nodes}
    graph.relations = new_relations
    graph.triplets  = new_triplets

    if verbose:
        print(f"Entities:  {before_nodes} -> {len(merged_nodes)}")
        print(f"Relations: {before_rels} -> {len(new_relations)}"
              f"  ({self_loops} self-loops dropped)")
        if merges:
            print(f"\nMerged {len(merges)} entity groups:")
            for display, variants in sorted(merges):
                print(f"  {display}  <-  {', '.join(variants)}")
        else:
            print("\nNo duplicate entities found.")

    return graph_store


resolve_entities(graph_store)
print("\n✅ Entity resolution complete")

Entities:  67 -> 57
Relations: 31 -> 31  (0 self-loops dropped)

Merged 8 entity groups:
  AI Act  <-  AI Act, Ai Act, EU AI Act
  ChatGPT  <-  ChatGPT, Chatgpt
  Fair Use  <-  Fair Use, Fair Use Doctrine
  Generative AI  <-  Generative AI, Generative Artificial Intelligence
  OpenAI  <-  OpenAI, Openai
  Thaler v. Perlmutter  <-  Thaler V. Perlmutter, Thaler v. Perlmutter
  U.S. Copyright Office  <-  Copyright Office, U.S. Copyright Office, US Copyright Office
  World Intellectual Property Organization  <-  WIPO, World Intellectual Property Organization

✅ Entity resolution complete


In [22]:
# Print out example of an article and the extracted entities & relationships
import copy

# Change this index to inspect a different article
sample = nodes[3]

print("=== Title ===")
print(sample.metadata.get("title", "untitled"))
print()
print("=== Raw text ===")
print(sample.text)
print()

# Re-run extraction on a copy so we don't mutate the original
extracted = await kg_extractor._aextract(copy.deepcopy(sample))

print("=== Extracted entities ===")
for node in extracted.metadata.get(KG_NODES_KEY, []):
    print(f"  [{node.label}] {node.name}")
    print(f"    {node.properties.get('entity_description', '')}")

print()
print("=== Extracted relationships ===")
id_to_name = {n.id: n.name for n in extracted.metadata.get(KG_NODES_KEY, [])}
for rel in extracted.metadata.get(KG_RELATIONS_KEY, []):
    src = id_to_name.get(rel.source_id, rel.source_id)
    tgt = id_to_name.get(rel.target_id, rel.target_id)
    print(f"  {src} --[{rel.label}]--> {tgt}")

=== Title ===
Intellectual Property Rights and AI-Generated Content ...

=== Raw text ===
Member-only story
Intellectual Property Rights and AI-Generated Content — Issues in Human Authorship, Fair Use Doctrine, and Output Liability
Critical IP Questions Every Leader Needs to Answer for Future Readiness — Analysis of Copyright Law Implications for Generative Artificial Intelligence — Comparative Legal Frameworks, Authorship Paradigms, and Training Data Considerations
Generative Artificial Intelligence is rapidly reshaping industries, but its collision with long-established copyright law presents critical strategic questions that leaders ignore at their peril. For leaders charting a course through the AI revolution, navigating the intricate maze of copyright law is no longer a niche legal concern but a core strategic imperative impacting innovation, risk, and global competitiveness.
In my discussions with leadership across various sectors, the rapid rise of generative AI consistently bri

In [23]:
# ── Print out unique entities extracted ───────────────────────────────────────────────────
from collections import defaultdict

by_type = defaultdict(list)
for node in graph_store.graph.nodes.values():
    if isinstance(node, EntityNode):
        by_type[node.label].append(node.name)

for entity_type, names in sorted(by_type.items()):
    print(f"\n{entity_type} ({len(names)})")
    for name in sorted(names):
        print(f"  {name}")


AI_SYSTEM (6)
  ChatGPT
  DABUS
  Generative AI
  Gpt Series
  Large Language Models
  Stable Diffusion

CONCEPT (3)
  Copyright
  Fair Use
  Intellectual Property

ENTITY (1)
  AI_SYSTEM

GOVERNMENT (7)
  Beijing Internet Court
  European Union
  Federal Trade Commission
  House Judiciary Committee Republicans
  U.S. Copyright Office
  US Patent and Trademark Office
  United States

LEGAL_CASE (4)
  Andersen v. Stability AI
  Bartz Case
  Thaler v. Perlmutter
  U.S. District Court For The District Of Columbia

LEGISLATION (11)
  AI Act
  Administrative Procedure Act
  Copyright Act
  Copyright Law
  Generative AI Copyright Disclosure Act of 2024
  No AI Fraud Act
  Part 1: Digital Replicas
  Part 2: Copyrightability
  Part 3: Generative AI Training
  Report On Copyright And Artificial Intelligence
  US Copyright Act of 1976

ORGANIZATION (14)
  Anthropic
  DeviantArt
  Disney
  Microsoft
  Midjourney
  National Institutes of Health
  Nixon Peabody
  OpenAI
  Stability AI
  The New Yo

In [24]:
# ── Inspect a specific untyped entity ─────────────────────────────────────────
# Extraction is LLM-driven, so exact entity names shift from run to run. Use the
# name below if it's present, otherwise fall back to whatever came back untyped
# (label "ENTITY") — the case this cell exists to examine.
inspect_name = "AI_SYSTEM"   # set to None to just take the first untyped entity

entities = [n for n in graph_store.graph.nodes.values() if isinstance(n, EntityNode)]
node = next((n for n in entities if n.name == inspect_name), None)

if node is None:
    if inspect_name is not None:
        print(f"⚠️  No entity named {inspect_name!r} — falling back to an untyped one.\n")
    node = next((n for n in entities if n.label == "ENTITY"), None)

if node is None:
    raise LookupError(
        f"No entity named {inspect_name!r}, and no untyped entities to fall back to. "
        f"Available names: {sorted(n.name for n in entities)}"
    )

print(f"Node:   {node.name!r}  label={node.label!r}")
print(f"Source: {node.properties.get('source', 'N/A')}")
print(f"Title:  {node.properties.get('title', 'N/A')}")
print()

# Look up the original article text by matching the title
title = node.properties.get("title")
article = next((n for n in nodes if n.metadata.get("title") == title), None)
if article:
    print("=== Article text ===")
    print(article.text)
print()

# Find all relations involving this entity
relations = [
    r for r in graph_store.graph.relations.values()
    if r.source_id == node.id or r.target_id == node.id
]
node_index = {
    n.id: n.name for n in graph_store.graph.nodes.values()
    if isinstance(n, EntityNode)
}
print(f"Relations ({len(relations)}):")
for r in relations:
    src = node_index.get(r.source_id, r.source_id)
    tgt = node_index.get(r.target_id, r.target_id)
    print(f"  {src} --[{r.label}]--> {tgt}")

Node:   'AI_SYSTEM'  label='ENTITY'
Source: Nixon Peabody
Title:  Generative AI: Navigating intellectual property

=== Article text ===
Generative AI is transforming creative and technology-driven industries at an unprecedented pace. While AI technologies have great potential to unlock new business opportunities and efficiencies, they also raise questions about authorship, ownership, and protecting human creativity.
These questions underscore a central legal dilemma: current intellectual property laws are struggling to keep pace with the unique challenges posed by AI-generated content. Traditional legal frameworks and precedents for copyright, patent, and trademark protection were conceived with human creators in mind, leaving uncertainty about how to protect, license, and enforce rights in works generated by, or with the assistance of, AI. Nixon Peabody serves as a forward-thinking partner for businesses exploring AI and intellectual property opportunities.
How does AI affect intellec

# 13. Build Communities and Generate Summaries

In [25]:
graph_store.build_communities()

summaries = graph_store.get_community_summaries()
print(f"\n✅ {len(summaries)} community summaries ready for querying")

Running community detection...
Graph has 57 nodes, 30 edges
Found 10 communities


/Users/prathameshnayak/Documents/projects/graphrag-ai-copyright/.venv/lib/python3.13/site-packages/graspologic/partition/leiden.py:607: UserWarning: Leiden partitions do not contain all nodes from the input graph because input graph contained isolate nodes.
  warnings.warn(


  Community 0: This cluster centers on major AI copyright disputes involving prominent technology developers like O...
  Community 1: This entity cluster centers on the legal conflict between major entertainment conglomerate Disney an...
  Community 2: This cluster centers on the U.S. Copyright Office, key legislative frameworks like the US Copyright ...
  Community 3: This cluster centers on the intersection of artificial intelligence, intellectual property, and glob...
  Community 4: This cluster centers on the intersection of **Generative AI** and **Large Language Models (LLMs)** w...
  Community 5: This cluster centers on the landmark intellectual property dispute involving the AI system **DABUS**...
  Community 6: This cluster centers on the legal debate surrounding copyright eligibility for AI-generated works, f...
  Community 7: This cluster centers on Edward Lee, a professor at Santa Clara University School of Law, and his cri...
  Community 8: This cluster centers on copyright

# 14. Visualise the Knowledge Graph

In [27]:
import json
import pathlib


def export_graph_data(graph_store: GraphRAGStore, output_file: str = "graph_data.json"):
    """
    Export the knowledge graph to a JSON file compatible with the D3.js template.
    Run this once after graph_store.build_communities().

    Saves to disk so you can re-run the visualization without reprocessing
    the full pipeline — which is expensive.
    """

    nx_graph = graph_store._to_networkx()

    # ── Build node metadata lookup ─────────────────────────────────────────
    node_meta = {}
    for node in graph_store.graph.nodes.values():
        if isinstance(node, EntityNode):
            node_meta[node.id] = {
                "id":          node.id,
                "label":       node.name,
                "type":        node.label if node.label else "OTHER",
                "description": node.properties.get("entity_description", ""),
            }

    # ── Nodes — only those present in the NetworkX graph ──────────────────
    nodes_data = [
        node_meta[n] for n in nx_graph.nodes() if n in node_meta
    ]

    # ── Links ──────────────────────────────────────────────────────────────
    links_data = []
    for source, target, data in nx_graph.edges(data=True):
        links_data.append({
            "source":      source,
            "target":      target,
            "label":       data.get("relationship", ""),
            "description": data.get("description",  ""),
        })

    # ── Assemble and write ─────────────────────────────────────────────────
    graph_data = {
        "nodes":       nodes_data,
        "links":       links_data,
        "communities": len(graph_store.get_community_summaries()),
    }

    pathlib.Path(output_file).write_text(json.dumps(graph_data, indent=2))

    print(f"✅ Graph data exported to '{output_file}'")
    print(f"   Nodes:       {len(nodes_data)}")
    print(f"   Edges:       {len(links_data)}")
    print(f"   Communities: {graph_data['communities']}")

    return graph_data


def visualize_graph(
    graph_store: GraphRAGStore = None,
    graph_data_file: str = "graph_data.json",
    template_file: str = "graph_template.html",
    output_file: str = GRAPH_OUTPUT_FILE,
):
    """
    Generate a D3.js interactive knowledge graph visualization.

    Can be called in two ways:

    1. Pass graph_store directly — exports fresh data then builds the HTML:
         visualize_graph(graph_store=graph_store)

    2. Load from a previously exported JSON file — no pipeline re-run needed:
         visualize_graph()  # loads graph_data.json automatically

    Features:
    - Color-coded nodes by entity type (from ontology)
    - Node size scales with degree (more connections = bigger)
    - Click a node to highlight its neighbourhood, dim everything else
    - Sidebar legend — click any entity type to toggle its visibility
    - Search bar to find and highlight any node by name
    - Sliders for link distance, charge strength, and edge label threshold
    - Hover tooltips with entity name, type, description, and connection count
    - Stats header showing total nodes, edges, and communities
    """

    # ── Get graph data ─────────────────────────────────────────────────────
    if graph_store is not None:
        # Export fresh data from the graph store and save to disk
        graph_data = export_graph_data(graph_store, graph_data_file)
    else:
        # Load from a previously saved JSON file
        data_path = pathlib.Path(graph_data_file)
        if not data_path.exists():
            print(f"⚠️  '{graph_data_file}' not found.")
            print(f"   Run export_graph_data(graph_store) first, or pass graph_store directly.")
            return
        graph_data = json.loads(data_path.read_text())

        print(f"✅ Loaded graph data from '{graph_data_file}'")
        print(f"   Nodes: {len(graph_data['nodes'])}  |  "
              f"Edges: {len(graph_data['links'])}  |  "
              f"Communities: {graph_data.get('communities', '—')}")

    # ── Load HTML template ─────────────────────────────────────────────────
    template_path = pathlib.Path(template_file)
    if not template_path.exists():
        print(f"⚠️  '{template_file}' not found.")
        print(f"   Make sure graph_template.html is in the same folder as this notebook.")
        return

    # ── Inject data into template and write output ─────────────────────────
    html = template_path.read_text()
    html = html.replace("GRAPH_DATA_PLACEHOLDER", json.dumps(graph_data))
    pathlib.Path(output_file).write_text(html)

    print(f"\n✅ Visualization saved to '{output_file}'")
    print(f"   Open it in your browser to explore.")


# Export data and build visualization from the graph store
visualize_graph(graph_store=graph_store)

✅ Graph data exported to 'graph_data.json'
   Nodes:       57
   Edges:       30
   Communities: 10

✅ Visualization saved to 'ai_copyright_graph.html'
   Open it in your browser to explore.
